# TinyLlama 1.1B – EGGROLL Integration Test

Validates the TinyLlama integration into HyperscaleES:
1. Setup & install
2. Parameter shape verification
3. Forward pass & top-k predictions
4. KV cache position tracking
5. Numerical comparison with HuggingFace
6. Greedy generation sanity check
7. EGGROLL training (fastzero)

**Requirements:** Colab with T4 GPU runtime (or local GPU).

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
import os
if os.path.exists("/content/HyperscaleES"):
    !cd /content/HyperscaleES && git pull
else:
    !git clone -b warming-up https://github.com/shr1ram/HyperscaleES.git /content/HyperscaleES
%cd /content/HyperscaleES
!pip install -e . -q

## 1. Load TinyLlama

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from functools import partial
import time

import hyperscalees as hs
from hyperscalees.models.llm.auto import get_model
from hyperscalees.models.common import simple_es_tree_key

print("JAX devices:", jax.devices())
print()

NOISER = hs.noiser.base_noiser.Noiser  # noop noiser for testing
base_model_key = jax.random.key(0)

print("Loading TinyLlama 1.1B...")
MODEL, full_params, tokenizer = get_model("tl1.1B", verbose=True)
config, params, scan_map, es_map = full_params
params = jax.device_put(params, jax.local_devices()[0])

frozen_noiser_params, noiser_params = NOISER.init_noiser(params, 0.0, None)
base_evo_keys = simple_es_tree_key(params, base_model_key, scan_map)

print("\nModel loaded successfully!")

## 2. Parameter Shape Verification

In [ ]:
expected_shapes = {
    "embed_tokens.weight":                (32000, 2048),
    "blocks.self_attn.q_proj.weight":      (22, 2048, 2048),
    "blocks.self_attn.k_proj.weight":      (22, 256, 2048),
    "blocks.self_attn.v_proj.weight":      (22, 256, 2048),
    "blocks.self_attn.o_proj.weight":      (22, 2048, 2048),
    "blocks.mlp.gate_proj.weight":         (22, 5632, 2048),
    "blocks.mlp.up_proj.weight":           (22, 5632, 2048),
    "blocks.mlp.down_proj.weight":         (22, 2048, 5632),
    "blocks.input_layernorm.weight":       (22, 2048),
    "blocks.post_attention_layernorm.weight": (22, 2048),
    "norm.weight":                         (2048,),
    "lm_head.weight":                      (32000, 2048),
}

def get_nested(d, dotted_key):
    for k in dotted_key.split("."):
        d = d[k]
    return d

all_ok = True
for name, expected in expected_shapes.items():
    actual = get_nested(params, name).shape
    status = "OK" if actual == expected else "FAIL"
    if status == "FAIL":
        all_ok = False
    print(f"  {status}  {name}: {actual} (expected {expected})")

print()
print("All shapes correct!" if all_ok else "SHAPE MISMATCHES FOUND")

## 3. Forward Pass & Top-k Predictions

In [ ]:
context = "The Eiffel tower is in the city of"
encoded = tokenizer.encode(context)
print(f"Input: '{context}'")
print(f"Tokens ({len(encoded)}): {encoded}")
print()

init_state = MODEL.default_state(params, config)
print(f"KV cache shape: {init_state['kv_cache'].shape}")
print(f"Initial cache_pos: {init_state['cache_pos']}")
print()

forward = partial(MODEL.forward, NOISER, frozen_noiser_params, noiser_params, config)

start = time.time()
out, state = jax.block_until_ready(forward(params, base_evo_keys, (0, 1), encoded, init_state))
elapsed = time.time() - start

print(f"Output shape: {out.shape} (expected ({len(encoded)}, 32000))")
print(f"Forward time: {elapsed:.3f}s")
print()

last_logits = out[-1]
soft_out = jax.nn.softmax(last_logits)
values, indices = jax.lax.top_k(soft_out, 10)

print("Top 10 next-token predictions:")
for i in range(10):
    print(f"  {values[i].item()*100:6.2f}%  {tokenizer.decode([indices[i].item()])!r}")

## 4. KV Cache Position Tracking

In [ ]:
print(f"cache_pos after prompt: {state['cache_pos']}  (expected {len(encoded)})")
assert state['cache_pos'] == len(encoded), "cache_pos mismatch!"

# Incremental single-token forward
next_tok = [int(indices[0])]
out2, state2 = jax.block_until_ready(forward(params, base_evo_keys, (0, 1), next_tok, state))

print(f"cache_pos after 1 more token: {state2['cache_pos']}  (expected {len(encoded) + 1})")
assert state2['cache_pos'] == len(encoded) + 1, "cache_pos mismatch after incremental step!"
assert out2.shape == (1, 32000), f"Expected (1, 32000), got {out2.shape}"

# Multi-step incremental
s = state2
for step in range(5):
    tok = [int(jnp.argmax(out2[-1]))]
    out2, s = jax.block_until_ready(forward(params, base_evo_keys, (0, 1), tok, s))

print(f"cache_pos after 6 more tokens: {s['cache_pos']}  (expected {len(encoded) + 7})")
assert s['cache_pos'] == len(encoded) + 7

print("\nKV cache tracking: PASS")

## 5. Numerical Comparison with HuggingFace

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer as HFAutoTokenizer

hf_model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.float32
)
hf_tok = HFAutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
hf_model.eval()

hf_input = hf_tok(context, return_tensors="pt")
with torch.no_grad():
    hf_output = hf_model(**hf_input)

hf_logits = hf_output.logits[0].numpy()  # (seq_len, vocab_size)

# Re-run JAX forward from scratch for fair comparison
jax_out, _ = jax.block_until_ready(forward(params, base_evo_keys, (0, 1), encoded, init_state))
jax_logits = np.array(jax_out.astype(jnp.float32))

print(f"HF  logits shape: {hf_logits.shape}")
print(f"JAX logits shape: {jax_logits.shape}")
assert hf_logits.shape == jax_logits.shape, "Shape mismatch!"
print()

In [ ]:
hf_last = hf_logits[-1]
jax_last = jax_logits[-1]

max_diff = np.max(np.abs(hf_last - jax_last))
mean_diff = np.mean(np.abs(hf_last - jax_last))
corr = np.corrcoef(hf_last, jax_last)[0, 1]

print(f"Max  abs diff: {max_diff:.6f}")
print(f"Mean abs diff: {mean_diff:.6f}")
print(f"Correlation:   {corr:.6f}")
print()

hf_top = int(np.argmax(hf_last))
jax_top = int(np.argmax(jax_last))
print(f"HF  top token: {hf_tok.decode([hf_top])!r}  (id={hf_top})")
print(f"JAX top token: {tokenizer.decode([jax_top])!r}  (id={jax_top})")
print()

if corr > 0.99 and hf_top == jax_top:
    print("Numerical match: PASS")
elif corr > 0.99:
    print(f"Numerical match: PARTIAL (correlation ok, top token differs – bf16 edge case)")
else:
    print(f"Numerical match: FAIL (correlation {corr:.4f} < 0.99)")

# Clean up HF model to free memory
del hf_model, hf_output
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 6. Greedy Generation Sanity Check

Generate 50 tokens greedily (temperature=0) to verify coherent output.

In [ ]:
prompt = "Once upon a time in a land far away,"
tokens = tokenizer.encode(prompt)
gen_state = MODEL.default_state(params, config)

out_logits, gen_state = jax.block_until_ready(
    forward(params, base_evo_keys, (0, 1), tokens, gen_state)
)

generated = list(tokens)
NUM_GENERATE = 50

for _ in range(NUM_GENERATE):
    next_id = int(jnp.argmax(out_logits[-1]))
    generated.append(next_id)
    out_logits, gen_state = jax.block_until_ready(
        forward(params, base_evo_keys, (0, 1), [next_id], gen_state)
    )

print(f"Prompt:    {prompt!r}")
print(f"Generated: {tokenizer.decode(generated)!r}")
print(f"\nTotal tokens: {len(generated)} ({len(tokens)} prompt + {NUM_GENERATE} generated)")
print(f"Final cache_pos: {gen_state['cache_pos']}")

## 7. EGGROLL Training (fastzero)

Run a short training loop to verify the full pipeline compiles and runs.

In [ ]:
# Configuration
MODEL_CHOICE = "tl1.1B"
TASK = "fastzero"
NUM_EPOCHS = 10
BATCH_SIZE = 256
SIGMA = 1e-3
LR_SCALE = 1.0
NOISER_NAME = "eggroll"

print(f"Model:   {MODEL_CHOICE}")
print(f"Noiser:  {NOISER_NAME}")
print(f"Task:    {TASK}")
print(f"Epochs:  {NUM_EPOCHS}")
print(f"Batch:   {BATCH_SIZE}")
print(f"Sigma:   {SIGMA}")
print(f"LR:      {LR_SCALE}")

In [ ]:
!python -m llm_experiments.general_do_evolution \
    --model_choice $MODEL_CHOICE \
    --task $TASK \
    --num_epochs $NUM_EPOCHS \
    --parallel_generations_per_gpu $BATCH_SIZE \
    --sigma $SIGMA \
    --lr_scale $LR_SCALE \
    --noiser $NOISER_NAME 2>&1 | tee tinyllama_train.log

In [ ]:
import re
import matplotlib.pyplot as plt

with open("tinyllama_train.log") as f:
    log = f.read()

avg_fit, lora_upd, nonlora_upd = [], [], []

for m in re.finditer(
    r"\tavg_fitness:\s*([-\d.e+]+).*?"
    r"\tlora_updates:\s*([-\d.e+]+).*?"
    r"\tnonlora_updates:\s*([-\d.e+]+)",
    log, re.DOTALL,
):
    avg_fit.append(float(m.group(1)))
    lora_upd.append(float(m.group(2)))
    nonlora_upd.append(float(m.group(3)))

epochs = list(range(len(avg_fit)))
print(f"Parsed {len(epochs)} epochs")

if epochs:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, avg_fit, "b-o", markersize=3)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Avg Fitness")
    ax1.set_title("TinyLlama EGGROLL – Fitness")
    ax1.grid(True)

    ax2.plot(epochs, lora_upd, label="LoRA updates")
    ax2.plot(epochs, nonlora_upd, label="Non-LoRA updates")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Update magnitude")
    ax2.set_title("Parameter Updates")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig("tinyllama_training.png", dpi=150)
    plt.show()
    print("Saved to tinyllama_training.png")
else:
    print("No training data found in log – check for errors above.")

## Summary

| Test | Status |
|------|--------|
| Parameter shapes | Cell 2 |
| Forward pass output shape | Cell 3 |
| KV cache tracking | Cell 4 |
| HuggingFace numerical match | Cells 5-6 |
| Greedy generation | Cell 7 |
| EGGROLL training | Cell 8-9 |